# 09 · Thesis figures (your figure layout)

Nothing is refitted: every figure reads the outputs of notebooks 02-06.

| Figure | Content | Folder |
|---|---|---|
| A | Spatial validation layout: blockCV blocks coloured by fold; purge circles for one outer fold | `private` (shows boreholes) |
| B | Model comparison: balanced accuracy, mean ± SD over the 5 outer folds | outputs |
| C | Confusion matrix (pooled outer test folds) and mean spatial ROC curve | outputs |
| D | Global SHAP importance, held-out permutation importance, SHAP beeswarm | outputs |

Figures C and D are drawn for the final model and for `EXTRA_MODELS` (default XGB).

**Run time:** under 1 minute.

In [ ]:
import json, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.toml").exists())))
import helpers as H

S = H.load_settings()
T0 = time.time()
print("QUICK TEST RUN - not thesis results" if S["run"]["quick"] else "FULL RUN", "| area:", S["area"]["area"])
import geopandas as gpd
import shap
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve

CELLS = "all"          # "all" (measured + filled test cells) or "measured"
FOLD = 1               # outer fold shown in the purge panel
EXTRA_MODELS = ["XGB"]

cells = pd.read_csv(H.processed_dir(S) / "cells.csv")
folds, block_m = H.load_folds(S, 1)
preds = pd.read_csv(H.out_dir(S, "03_spatial_cv", private=True) / "outer_predictions.csv")
summ = pd.read_csv(H.out_dir(S, "04_results") / "scores_mean_sd.csv")
final_model = json.loads((H.out_dir(S, "05_final_model") / "final_model.json").read_text())["model"]
if CELLS == "measured":
    preds = preds[preds.measured == 1]
summ = summ[summ.cells == CELLS]
main = preds[(preds.multiple == 1) & (preds.purge_x == 0)]
OUT, PRIV = H.out_dir(S, "09_thesis_figures"), H.out_dir(S, "09_thesis_figures", private=True)
B = f"{block_m:.0f}"

MAGENTA, WINE, TEAL, YELLOW, NAVY, GREY = "#A0004E", "#6E0032", "#17918C", "#E5B74A", "#17365C", "#9E9E9E"
SEQ = [MAGENTA, WINE, TEAL, YELLOW, NAVY]
FOLD_COL = ["#2C6FBB", "#F28E2B", "#59A14F", "#C0000B", "#7B5CA8"]
CMAP = LinearSegmentedColormap.from_list("navy_magenta", [NAVY, "#D9A6C1", MAGENTA])
NICE = {"RF": "Random Forest", "XGB": "XGBoost", "MLP": "MLP", "GWRF": "GWRF", "GWXGB": "GWXGBoost",
        "DEPTH_ONLY": "Depth only\n(reference)"}
plt.rcParams.update({"font.family": "sans-serif", "axes.spines.top": False, "axes.spines.right": False,
                     "figure.facecolor": "white", "savefig.dpi": 300, "savefig.bbox": "tight"})

def save(fig, folder, name):
    for ext in ("png", "pdf"):
        fig.savefig(folder / f"{name}.{ext}")
    print("saved", folder / f"{name}.png")

## Fig. A · Spatial validation layout

In [ ]:
blk = gpd.read_file(H.out_dir(S, "02_blockcv", private=True) / "blocks_x1.gpkg")
bh = cells.groupby("BH_Code")[["UTM_Easting_m", "UTM_Northing_m"]].first()
bh["fold"] = bh.index.map(folds).astype(int)
pts = gpd.points_from_xy(bh.UTM_Easting_m, bh.UTM_Northing_m)
occupied = blk[blk.intersects(gpd.GeoSeries(pts, crs=blk.crs).union_all())]
km = lambda v, _: f"{v / 1000:.0f}"
fig, axes = plt.subplots(1, 2, figsize=(18, 7.2))
for ax in axes:
    for k in sorted(bh.fold.unique()):
        occupied[occupied.folds == k].plot(ax=ax, color=FOLD_COL[k - 1], alpha=0.22, edgecolor=FOLD_COL[k - 1], linewidth=0.6)
        b = bh[bh.fold == k]
        ax.scatter(b.UTM_Easting_m, b.UTM_Northing_m, s=42, color=FOLD_COL[k - 1], edgecolor="white", linewidth=0.8,
                   label=f"Fold {k}", zorder=3)
    ax.xaxis.set_major_formatter(km); ax.yaxis.set_major_formatter(km)
    ax.set_xlabel("UTM Easting (km)", fontsize=12); ax.set_ylabel("UTM Northing (km)", fontsize=12)
    ax.set_aspect("equal")
    for s_ in ax.spines.values():
        s_.set_visible(True); s_.set_color("#666666")
axes[0].set_title("Spatial Validation Layout\n", fontsize=16, fontweight="bold")
axes[0].text(0.5, 1.01, f"{B} m spatial blocks (blockCV, k = 5)", transform=axes[0].transAxes, ha="center", fontsize=11, color="#555555")
axes[0].legend(frameon=False, loc="upper left", bbox_to_anchor=(1.0, 1.0))
test, train = bh[bh.fold == FOLD], bh[bh.fold != FOLD]
d = H.nearest_distance(train[["UTM_Easting_m", "UTM_Northing_m"]].values, test[["UTM_Easting_m", "UTM_Northing_m"]].values)
removed = train[d < block_m]
for x, y in test[["UTM_Easting_m", "UTM_Northing_m"]].values:
    axes[1].add_patch(plt.Circle((x, y), block_m, fill=False, color="#888888", linewidth=0.9, zorder=2))
axes[1].scatter(removed.UTM_Easting_m, removed.UTM_Northing_m, s=70, marker="x", color="black", linewidth=1.4, zorder=4)
axes[1].set_title(f"Spatial Purge Layout (outer fold {FOLD})\n", fontsize=16, fontweight="bold")
axes[1].text(0.5, 1.01, f"training boreholes within {B} m of a fold-{FOLD} test borehole removed",
             transform=axes[1].transAxes, ha="center", fontsize=11, color="#555555")
h, l = axes[0].get_legend_handles_labels()
axes[1].legend(h + [Line2D([], [], color="#888888", lw=0.9), Line2D([], [], marker="x", ls="", color="black")],
               l + [f"{B} m purge circle", f"removed training borehole ({len(removed)})"],
               frameon=False, loc="upper left", bbox_to_anchor=(1.0, 1.0))
fig.tight_layout()
save(fig, PRIV, "figA_spatial_validation_layout")

## Fig. B · Model comparison

In [ ]:
s = summ[(summ.multiple == 1) & (summ.purge_x == 0)].sort_values("balanced_accuracy_mean", ascending=False)
ml = s[s.model != H.DEPTH_ONLY]
order = list(ml.model) + [H.DEPTH_ONLY]
s = s.set_index("model").loc[order]
best_mean = ml.sort_values("macro_f1_mean", ascending=False).model.iloc[0]
fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(order)); sd = s.balanced_accuracy_sd.fillna(0)
ax.bar(x, s.balanced_accuracy_mean, color=[SEQ[i % 5] for i in range(len(ml))] + [GREY], width=0.82,
       yerr=sd, capsize=4, error_kw={"elinewidth": 1, "ecolor": "#444444"})
for xi, v in zip(x, s.balanced_accuracy_mean):
    ax.text(xi, v + sd.max() + 0.01, f"{v:.4f}", ha="center", fontsize=11)
mark = lambda m: (" *" if m == final_model else "") + (" †" if m == best_mean and m != final_model else "")
ax.set_xticks(x, [NICE.get(m, m) + mark(m) for m in order], fontsize=12)
ax.set_ylim(0, 0.8); ax.set_ylabel("Balanced Accuracy", fontsize=13)
ax.set_title("Model Comparison — Spatial Cross-Validation", fontsize=15)
fig.text(0.5, -0.02, f"Mean over 5 outer spatial test folds ({B} m blocks); error bars = SD; * = final model"
         + ("; † = highest mean Macro-F1" if best_mean != final_model else "") + f"; cells: {CELLS}", ha="center", fontsize=10)
save(fig, OUT, "figB_model_comparison")
display(s[["balanced_accuracy_mean", "balanced_accuracy_sd", "macro_f1_mean", "roc_auc_mean"]].round(4))

## Fig. C · Confusion matrix and mean spatial ROC curve

In [ ]:
role = lambda m: "final model" if m == final_model else ("highest mean Macro-F1, not selected" if m == best_mean else "not selected")
MODELS = [final_model] + [m for m in EXTRA_MODELS if m != final_model and m in set(main.model)]
for model in MODELS:
    f = main[main.model == model]
    cm = confusion_matrix(f.y, f.y_pred, labels=[0, 1])
    fig, axes = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={"width_ratios": [1.1, 1]})
    ax = axes[0]
    im = ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]}", ha="center", va="center", fontsize=16,
                    color="white" if cm[i, j] > (cm.min() + cm.max()) / 2 else NAVY)
    ax.set_xticks([0, 1], ["Sand", "Clay"], fontsize=13); ax.set_yticks([0, 1], ["Sand", "Clay"], fontsize=13)
    ax.xaxis.tick_top(); ax.xaxis.set_label_position("top")
    ax.set_xlabel("Prediction", fontsize=13); ax.set_ylabel("Actual", fontsize=13)
    ax.set_title(f"Confusion Matrix — {NICE.get(model, model)}\n({role(model)})", fontsize=16, pad=48)
    fig.colorbar(im, ax=ax, fraction=0.046)
    ax = axes[1]
    grid = np.linspace(0, 1, 201); tprs, aucs = [], []
    for k, g in f.groupby("outer_fold"):
        if g.y.nunique() == 2:
            fpr, tpr, _ = roc_curve(g.y, g.p_clay)
            tprs.append(np.interp(grid, fpr, tpr)); tprs[-1][0] = 0.0
            aucs.append(roc_auc_score(g.y, g.p_clay))
    mean_tpr = np.mean(tprs, axis=0); mean_tpr[-1] = 1.0
    ax.plot(grid, mean_tpr, color=NAVY, linewidth=2.2, label=f"Mean AUC = {np.mean(aucs):.4f}")
    ax.plot([0, 1], [0, 1], linestyle="--", color="#777777")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
    ax.set_xlabel("False positive rate", fontsize=12); ax.set_ylabel("True positive rate", fontsize=12)
    ax.set_title(f"{NICE.get(model, model)} — Mean Spatial ROC Curve ({B} m blocks)", fontsize=12)
    ax.legend(frameon=False, loc="upper left", fontsize=11)
    fig.tight_layout()
    save(fig, OUT, f"figC_confusion_roc_{model}")

## Fig. D · Explainable AI

In [ ]:
sv_all = pd.read_csv(H.out_dir(S, "06_xai", private=True) / "shap_values_by_cell.csv")
perm_all = pd.read_csv(H.out_dir(S, "06_xai") / "permutation_importance.csv")
feat = H.FEATURES
short = {"UTM_Easting_m": "Easting", "UTM_Northing_m": "Northing", "Top_Elevation_m": "Elevation"}
for model in [m for m in MODELS if m in set(sv_all.model)]:
    sv = sv_all[sv_all.model == model]
    if CELLS == "measured":
        sv = sv[sv.measured == 1]
    vals = sv[[f"shap_{short[c]}" for c in feat]].values
    perm = perm_all[(perm_all.model == model) & (perm_all.kind == "single") & (perm_all.cells == CELLS)]
    perm = perm.set_index("input").reindex([short[c] for c in feat])
    fig = plt.figure(figsize=(13, 9.4))
    ax1 = fig.add_axes([0.08, 0.56, 0.38, 0.34]); ax2 = fig.add_axes([0.58, 0.56, 0.38, 0.34])
    g = pd.Series(np.abs(vals).mean(axis=0), index=feat).sort_values()
    ax1.barh(g.index, g.values, color=MAGENTA); ax1.set_xlabel("Mean absolute SHAP value")
    ax1.set_title("Global SHAP importance", fontsize=11)
    p = perm["mean"].rename(index={short[c]: c for c in feat}).sort_values()
    ax2.barh(p.index, p.values, color=MAGENTA); ax2.set_xlabel("Decrease in balanced accuracy")
    ax2.set_title("Held-out permutation importance", fontsize=11)
    ax3 = fig.add_axes([0.2, 0.06, 0.62, 0.40])
    plt.sca(ax3)
    shap.plots.beeswarm(shap.Explanation(values=vals, base_values=sv.base.values, data=sv[feat].values,
                                         feature_names=feat), color=CMAP, show=False, plot_size=None)
    ax3.set_title("SHAP beeswarm", fontsize=15)
    fig.suptitle(f"{NICE.get(model, model)} ({role(model)})", fontsize=14, y=0.98)
    save(fig, OUT, f"figD_xai_{model}")
print(f"Finished in {time.time() - T0:.0f} s.")